# Clinical AI Portfolio — Cumulative Step-by-Step Implementation

This is the single cumulative implementation notebook for the project. Future phases should append dated sections here rather than create a second implementation log.

**Recorded on:** 2026-08-19  
**Completed here:** dataset snapshot, Phase 0 (governance and clinical question), and Phase 1 (reproducible project foundation).  
**Not started:** cohort transformation, EDA, modeling, API, web, Android, or deployment.

> Educational research only. Nothing in this project is a diagnosis, medical advice, clinically validated software, or a medical device.

## 1. Decisions made before implementation

- **Dataset:** CDC/NCHS NHANES 2017–March 2020 pre-pandemic public-use combined sample.
- **Task:** estimate a research probability for **current** diabetes status; never call it future diabetes risk.
- **Population:** adults age 20 or older represented by the survey, initially excluding participants recorded as pregnant at examination.
- **Provisional outcome:** doctor-diagnosed diabetes (`DIQ010 = 1`) or HbA1c (`LBXGH`) ≥ 6.5%. This must pass Phase 2 codebook and feasibility review before it is frozen.
- **Prediction time:** a hypothetical non-invasive screening encounter before outcome fields or laboratory results are made available to the model.
- **Candidate predictors:** age, sex, BMI, waist, averaged measured blood pressure, physical activity, and smoking. Race/ethnicity is initially reserved for subgroup evaluation.
- **Prohibited predictors:** `DIQ010`, `LBXGH`, treatment variables, and any post-outcome/downstream information.
- **Users:** portfolio reviewers, students, data scientists, and software engineers using synthetic demo inputs.
- **Non-use:** diagnosis, treatment, triage, prescribing, replacing laboratory tests, emergencies, clinical integration, or individual care decisions.

Full decisions are in `docs/governance/project_charter.md`, `clinical_question.md`, `risk_register.md`, and `acceptance_gates.md`.

## 2. Why this dataset and cycle

CDC combined the incomplete 2019–March 2020 collection with 2017–2018 to create the nationally representative 2017–March 2020 pre-pandemic public-use sample. Analyses making population claims must use the combined-cycle design variables and appropriate weights; the incomplete 2019–2020 portion must not be treated independently as nationally representative.

Official references:

- [Demographics and combined-sample weights](https://wwwn.cdc.gov/nchs/nhanes/Search/DataPage.aspx?Component=Demographics&Cycle=2017-2020)
- [Examination components](https://wwwn.cdc.gov/Nchs/Nhanes/Search/DataPage.aspx?Component=Examination&Cycle=2017-2020)
- [Laboratory components](https://wwwn.cdc.gov/nchs/nhanes/Search/DataPage.aspx?Component=Laboratory&Cycle=2017-2020)
- [Diabetes questionnaire documentation](https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2017/DataFiles/P_DIQ.htm)

The files are public-use and de-identified, but they are not relicensed by this repository. Raw and derived participant-level files remain outside Git and will never be served by the demo.

## 3. Workspace creation command

The following PowerShell command created clean boundaries for raw/interim/processed data, source, tests, governance, CI, and future deployment artifacts:

```powershell
$folders = @(
  'data/raw/nhanes_2017_2020', 'data/interim', 'data/processed',
  'docs/governance', 'src/clinical_ml', 'tests', 'configs',
  'deployment/api', 'deployment/model', '.github/workflows'
)
$folders | ForEach-Object { New-Item -ItemType Directory -Force -Path $_ | Out-Null }
```

`deployment/model/` and `deployment/api/` are deliberately clean artifact directories. Future generated/promoted artifacts will be isolated there and ignored by Git unless a non-sensitive manifest is explicitly allow-listed.

## 4. Dataset download command

Seven official public-use XPT components were downloaded directly from CDC/NCHS:

```powershell
$base = 'https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2017/DataFiles'
$files = @(
  'P_DEMO.xpt', 'P_DIQ.xpt', 'P_GHB.xpt', 'P_BMX.xpt',
  'P_BPXO.xpt', 'P_PAQ.xpt', 'P_SMQ.xpt'
)
foreach ($file in $files) {
  Invoke-WebRequest -Uri "$base/$file" `
    -OutFile (Join-Path 'data/raw/nhanes_2017_2020' $file)
}
```

A reusable and idempotent version is saved as `scripts/download_nhanes.ps1`. It skips existing files and always verifies SHA-256.

## 5. Frozen source manifest and verification results

| File | Bytes | Rows | Columns | SHA-256 |
|---|---:|---:|---:|---|
| P_BMX.xpt | 2,520,640 | 14,300 | 22 | `7038d0da3169420a4a3cbaec09ac586a701b38e1c7b925431b60e13ac24fed66` |
| P_BPXO.xpt | 1,039,840 | 11,656 | 12 | `5edbcb81e99fe8dad38deb8b642570278b200c525d88ffa29ae3ceb4dd38bde4` |
| P_DEMO.xpt | 3,614,720 | 15,560 | 29 | `2e46c6c26bf77cd8989f64011ace12cbf42c0f3e03414eb59acc5328c8f87913` |
| P_DIQ.xpt | 3,361,520 | 14,986 | 28 | `79153f799fc2171792771c4aa250029c62807f14915f1f81405b03233b1e5ae3` |
| P_GHB.xpt | 167,600 | 10,409 | 2 | `dac9e423f56041c2ec46486cb16be16d3347e0a81a575985882ffad5a60e1195` |
| P_PAQ.xpt | 1,321,440 | 9,693 | 17 | `d0e13cebf96181949e983ab58d14c58689e801e8711d366d2b5daab3c7b9ee65` |
| P_SMQ.xpt | 1,428,560 | 11,137 | 16 | `29b7f6c59ab570c042c866312f0eb4329009e85fadbda6f9427a8bf31ecb3a3f` |

All seven files parsed successfully with pandas and contained the participant key `SEQN`. Hashes are tracked in `data/raw/nhanes_2017_2020/manifest.sha256`; XPT contents are ignored.

In [ ]:
# Optional reproducibility check: run after the raw snapshot is downloaded.
import hashlib
from pathlib import Path

import pandas as pd

root = Path("data/raw/nhanes_2017_2020")
for line in (root / "manifest.sha256").read_text().splitlines():
    expected, name = line.split(maxsplit=1)
    path = root / name
    actual = hashlib.sha256(path.read_bytes()).hexdigest()
    frame = pd.read_sas(path, format="xport")
    assert actual == expected and "SEQN" in frame.columns
    print(f"{name}: {len(frame):,} rows, {len(frame.columns)} columns, verified")

## 6. Phase 0 implementation — governance and clinical question

Created artifacts:

| File | Purpose |
|---|---|
| `docs/governance/project_charter.md` | Intended use, non-use, population, prediction time, outcome, candidate and prohibited features |
| `docs/governance/clinical_question.md` | Research question, estimand, unit, time zero, evaluation targets |
| `docs/governance/risk_register.md` | Harms, controls, and release consequences |
| `docs/governance/acceptance_gates.md` | Before-modeling, promotion, and deployment gates |
| `docs/data_source_decision.md` | Provenance, frozen components, terms, and limitations |
| `configs/project.yaml` | Machine-readable scope, cohort, outcome, prohibited predictors, artifact paths |

Key safety design: the label fields and treatment variables are explicitly banned as predictors, cross-sectional output is labeled as current status, and no numeric model-performance threshold was invented before feasibility analysis. Quantitative gates will be frozen after Phase 2 cohort analysis and before candidate comparison.

## 7. Phase 1 implementation — reproducible foundation

Created foundation:

- `pyproject.toml`: package metadata and exact Python/dev dependency pins.
- `.python-version`: Python 3.14.2.
- `.pre-commit-config.yaml`: Ruff lint/fix and formatting hooks.
- `src/clinical_ml/`: concise typed package skeleton and shared safety metadata.
- `tests/test_foundation.py`: safety metadata, raw-data exclusion, and deployment-boundary checks.
- `.github/workflows/ci.yml`: lint, format, type checks, tests, and notebook validation.
- `.github/pull_request_template.md` and issue template: quality/safety/reproducibility checks.
- `README.md`, `CONTRIBUTING.md`, `SECURITY.md`, `docs/definition_of_done.md`, and `CHANGELOG.md`.
- MIT `LICENSE`, `NOTICE`, and `CITATION.cff`; dataset terms remain separate.
- `.gitignore`: excludes raw/derived data, model/API artifacts, secrets, environments, and build output.

Current pinned direct versions: pandas 3.0.5, scikit-learn 1.9.0, pytest 9.1.1, nbformat 5.11.0, Ruff 0.16.0, mypy 2.3.0, and pre-commit 4.6.1. Transitive dependencies are resolved in the isolated environment; a hash-locked cross-platform dependency file and SBOM are release-stage deliverables.

## 8. Environment and version-control commands

```powershell
python -m venv .venv
.\.venv\Scripts\python.exe -m pip install --upgrade pip
.\.venv\Scripts\python.exe -m pip install -e ".[dev]"

git init -b main
.\.venv\Scripts\pre-commit.exe install

git check-ignore `
  data/raw/nhanes_2017_2020/P_DEMO.xpt `
  deployment/model/example.joblib `
  deployment/api/example.tar.gz
```

Observed result: Git initialized on `main`, the pre-commit hook was installed, and all three sensitive/generated example paths were correctly ignored. No remote or commit was created. Remote branch protection remains intentionally pending until a repository host is selected.

## 9. Quality commands and results

```powershell
.\.venv\Scripts\ruff.exe check .
.\.venv\Scripts\ruff.exe format --check .
.\.venv\Scripts\mypy.exe src
.\.venv\Scripts\pytest.exe
.\.venv\Scripts\python.exe -c "from pathlib import Path; import nbformat; paths=list(Path('.').glob('*.ipynb'))+list(Path('notebooks').glob('*.ipynb')); [nbformat.read(p, as_version=4) for p in paths]; print(f'validated {len(paths)} notebook(s)')"
```

First run: lint and types passed; 3 tests passed; the roadmap notebook validated. Ruff reported three source/test files requiring formatting. Corrective command:

```powershell
.\.venv\Scripts\ruff.exe format src tests
```

After this notebook was created, the first final lint pass caught one import-order issue in its reproducibility cell (`hashlib` needed to precede `Path`). The order was corrected immediately. The repeated full gate passed: Ruff lint passed; all 19 checked files were formatted; mypy found no issues; 3 tests passed; both notebooks validated; and all 7 dataset checksums verified. A final strict notebook-only pass validated both notebooks without warnings after adding stable cell IDs.

## 10. Copyright, licensing, data, and privacy controls implemented

- Original repository code/documentation use the MIT License with `Copyright (c) 2026 Clinical AI Portfolio Contributors`.
- `NOTICE` separates the repository license from NHANES data terms and disclaims CDC/NCHS endorsement.
- Raw and derived participant-level data are excluded from Git. Only source metadata, checksums, and a downloader are tracked.
- Model/API build artifacts, credentials, signing keys, local environments, and notebook checkpoints are excluded.
- `SECURITY.md` prohibits real identifiers and raw health-input logging.
- `CITATION.cff` exists, with its repository URL intentionally left as a clearly marked placeholder until a remote exists.

The copyright-holder label should be updated if the owner wants a personal or organizational legal name before the first public release.

## 11. Phase status and next gate

| Item | Status | Evidence |
|---|---|---|
| Dataset downloaded | Complete | 7 local XPT files; manifest and parse checks |
| Phase 0 | Complete | Governance, clinical question, risks, gates, data decision |
| Phase 1 local foundation | Complete | Git, pinned environment, hooks, CI, tests, license/docs |
| Hosted remote/branch protection | Deferred | Requires selection/authorization of repository host |
| Phase 2 data pipeline | Complete | Validated cohort builder, cohort flow, dictionary, processed local table |
| Phase 3 EDA and splits | Complete | Aggregate analysis, train-only ranking, 70/15/15 split, app input schema |
| Model/API/web/Android | Not started | Awaiting Phase 4 and later phases |

**Next approved gate:** Phase 4 baseline modeling, candidate comparison, calibration, subgroup evaluation, and locked-test evaluation.

# Phase 2 and Phase 3 — Preprocessing, EDA, Splits, and App Features

**Implemented:** 2026-08-19. Phase 2 converts seven immutable XPT components into a validated adult analytical cohort. Phase 3 performs aggregate EDA, creates deterministic leakage-safe splits, and ranks candidate app inputs using training data only. No final predictive model is trained here.

## 12. Reproducible command and outputs

```powershell
.\.venv\Scripts\python.exe scripts\build_phase2_3.py
```

This command reads the verified raw snapshot, validates one-to-one `SEQN` joins, applies the cohort rules, derives features, assigns splits, performs train-only mutual-information ranking, and writes:

- `data/processed/modeling_cohort.csv` — local participant-level modeling table; Git-ignored.
- `data/processed/feature_ranking.csv` — local ranking output; Git-ignored.
- `configs/app_features.json` — tracked provisional UI/API input contract.
- `reports/figures/phase3/*.png` — aggregate, non-participant-level EDA figures.
- `docs/data_dictionary.md`, `docs/phase2_3_report.md`, and `docs/app_input_spec.md`.

## 13. Phase 2 method and cohort flow

The preliminary outcome definition was tightened during implementation. A participant is positive when doctor-diagnosed diabetes is reported or HbA1c is at least 6.5%. A negative requires a negative/borderline diabetes response **and** observed HbA1c below 6.5%. This prevents missing HbA1c from being silently treated as normal.

| Cohort step | Remaining |
|---|---:|
| Source demographics | 15,560 |
| Adults age 20+ | 9,232 |
| After recorded-pregnancy exclusion | 9,145 |
| Positive combined MEC weight | 9,145 |
| Unambiguous outcome | 8,160 |
| Positive outcome | 1,701 |

The 985 ambiguous cases were excluded. Every source component had a unique `SEQN`; joins were validated one-to-one. Outcome fields and treatment variables are excluded from predictors. Questionnaire skip patterns and refused/don't-know codes are recoded rather than treated numerically.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

processed = Path("data/processed")
cohort = pd.read_csv(processed / "modeling_cohort.csv")
ranking = pd.read_csv(processed / "feature_ranking.csv")

split_summary = cohort.groupby("split")["outcome"].agg(["count", "sum", "mean"])
weighted_prevalence = np.average(cohort["outcome"], weights=cohort["WTMECPRP"])
print(split_summary)
print(f"Unweighted prevalence: {cohort['outcome'].mean():.2%}")
print(f"Survey-weighted prevalence: {weighted_prevalence:.2%}")

## 14. Phase 3 split design

| Split | N | Positive | Positive rate |
|---|---:|---:|---:|
| Train | 5,712 | 1,191 | 20.85% |
| Validation | 1,224 | 255 | 20.83% |
| Test | 1,224 | 255 | 20.83% |

The deterministic participant-level split uses seed `20260819`. Only training data were used to fill temporary medians for mutual-information ranking. The test split stays locked for Phase 4 final evaluation. This is an internal random split, not external or temporal validation; that limitation must remain visible.

In [ ]:
selected = ranking.head(6)["feature"].tolist()
missing = cohort[selected].isna().mean().mul(100).round(2)
means = cohort.groupby("outcome")[selected].mean().round(2).T
support = cohort[selected].quantile([0.01, 0.50, 0.99]).round(2).T

display(ranking.round(5))
display(missing.rename("missing_percent"))
display(means.rename(columns={0: "outcome_0", 1: "outcome_1"}))
display(support.rename(columns={0.01: "p01", 0.5: "median", 0.99: "p99"}))

## 15. Aggregate EDA figures

![Outcome prevalence](reports/figures/phase3/outcome_prevalence.png)

![Candidate feature missingness](reports/figures/phase3/feature_missingness.png)

![Training-only feature ranking](reports/figures/phase3/feature_ranking.png)

Unweighted outcome prevalence is 20.85%; survey-weighted prevalence is 14.56%. Selected-feature missingness ranges from 0% to 10.91%. Population estimates and predictive-sample summaries answer different questions and remain labeled separately.

## 16. Feature selection result

| Rank | App/model feature | Training MI | Missing | UI entry |
|---:|---|---:|---:|---|
| 1 | Age | 0.05881 | 0.00% | Age stepper |
| 2 | Waist circumference | 0.04228 | 6.00% | Decimal cm field |
| 3 | BMI | 0.03877 | 2.87% | Derived visibly from height and weight |
| 4 | Physical activity | 0.00872 | 0.00% | Yes/no segmented control |
| 5 | Diastolic BP | 0.00716 | 10.91% | Paired BP field |
| 6 | Systolic BP | 0.00702 | 10.91% | Paired BP field |

Sex and current smoking ranked seventh and eighth and are not included in the provisional app model. Mutual information is not causal importance or final model evidence. BMI/waist correlation is 0.905; systolic/diastolic correlation is 0.604. Phase 4 must compare ablations and regularized models before freezing the production schema.

## 17. APP graphical-interface pre-design

The user enters seven values for six model features because BMI is derived and shown from height plus weight.

```text
┌──────────────────────────────────────────┐
│ Educational diabetes screening demo     │
│ Not a diagnosis or medical advice       │
├──────────────────────────────────────────┤
│ Age [  ] years                           │
│ Height [   ] cm   Weight [   ] kg        │
│ Calculated BMI: -- kg/m²                 │
│ Waist circumference [    ] cm            │
│ Blood pressure [SYS] / [DIA] mmHg        │
│ Regular ≥10-minute activity? Yes / No    │
├──────────────────────────────────────────┤
│ [ Review inputs ]       [ Estimate ]     │
└──────────────────────────────────────────┘
```

Use numeric mobile keyboards, visible units, no default activity answer, inline range errors, a review step, and persistent safety text. Inputs are not stored by default. `configs/app_features.json` is the provisional machine-readable contract; `docs/app_input_spec.md` contains the full interaction rules. The result screen waits for Phase 4 calibration and threshold decisions.

## 18. Implementation checks and corrections

- A prototype initially allowed negative self-report with missing HbA1c to become negative. The final pipeline excludes those ambiguous cases.
- The first pipeline run exposed an incomplete UI mapping when diastolic BP entered the top six. The schema was corrected to support both BP values.
- Feature ranking uses training data only. Median filling used for ranking is fit on training rows and is not the Phase 4 modeling pipeline.
- Processed participant rows remain Git-ignored; only aggregate figures, documentation, configuration, and code are tracked.
- Tests cover smoking skip patterns, activity derivation, disjoint stratified splits, and UI-schema coverage.
- Final gate: Ruff passed; 26 files were formatted; strict mypy passed for 4 source files; 7 tests passed; both notebooks validated; all 7 raw checksums matched; and all 3 aggregate figures passed visual inspection.

## 19. Next gate

Phase 4 and Phase 5 are completed below. The next gate is Phase 6 web application implementation against the frozen OpenAPI and app-feature contracts.

# Phase 4 and Phase 5 — Model Selection, Evaluation, Packaging, and API

**Implemented:** 2026-08-19. Model choices and feature ablations used training and validation data only. The locked test set was evaluated once after the candidate, feature set, hyperparameter, selection rule, and threshold rule were frozen.

## 20. Reproducible commands

```powershell
.\.venv\Scripts\python.exe scripts\train_phase4.py
.\.venv\Scripts\python.exe -m scripts.export_openapi
.\.venv\Scripts\uvicorn.exe deployment.api.app:app --host 127.0.0.1 --port 8000
```

`train_phase4.py` compares candidates, chooses the validation winner, evaluates the test split once, writes aggregate metrics and the model card inputs, creates the evaluation figure, packages the complete pipeline in `deployment/model`, and writes a provenance manifest.

In [ ]:
import json
from pathlib import Path

metrics = json.loads(Path("reports/model_metrics.json").read_text())
comparison = metrics["candidate_comparison"]
print("Selected:", metrics["selected_candidate"])
print("Features:", metrics["features"])
print("Weighted test:", metrics["test_weighted"])

## 21. Validation candidate comparison

| Candidate | Features | Weighted ROC-AUC | Weighted PR-AUC | Weighted Brier |
|---|---:|---:|---:|---:|
| Logistic: compact diastolic | 4 | 0.7716 | 0.3245 | **0.1047** |
| Logistic: no diastolic | 5 | 0.7709 | 0.3119 | 0.1052 |
| Logistic: compact systolic | 4 | 0.7711 | 0.3109 | 0.1053 |
| Logistic: six features | 6 | 0.7722 | 0.3094 | 0.1054 |
| Logistic: no BMI | 5 | **0.7724** | 0.3084 | 0.1054 |
| Histogram gradient boosting | 6 | 0.7675 | 0.3183 | 0.1067 |
| Prevalence baseline | 6 | 0.5000 | 0.1356 | 0.1173 |

Prespecified selection rule: among candidates within 0.005 weighted validation ROC-AUC of the best, choose the lowest Brier score, then fewer features. The selected regularized logistic pipeline (`C=0.01`) uses age, waist circumference, physical activity, and diastolic BP. This reduces the app from seven entered values in the Phase 3 draft to four.

## 22. Locked-test evaluation and threshold

| Evaluation | ROC-AUC | PR-AUC | Brier | Log loss |
|---|---:|---:|---:|---:|
| Test, survey-weighted | 0.7710 | 0.3960 | 0.1171 | 0.3764 |
| Test, unweighted | 0.7742 | 0.4312 | 0.1449 | 0.4420 |

Unweighted participant-bootstrap 95% intervals: ROC-AUC 0.744–0.803 and Brier 0.1326–0.1590. These are not complex-survey confidence intervals.

Threshold 0.1301 was selected on weighted validation data as the lowest-false-positive point reaching at least 80% sensitivity. Validation sensitivity/specificity were 0.801/0.628; weighted test sensitivity/specificity were 0.771/0.656. The sensitivity target did not fully transport, so the API names the output `above_validation_threshold` and never describes it as diagnosis.

## 23. Model evaluation figure

![ROC and calibration](reports/figures/phase4/model_evaluation.png)

The plot is descriptive. Subgroup inspection found unweighted test ROC-AUC 0.622 in adults 60+, materially below the overall result. Race/ethnicity subgroup sizes ranged from 50 to 418. These results do not demonstrate clinical validity or fairness and are prominent in `docs/model_card.md`.

## 24. Model package and provenance

- Local artifact: `deployment/model/clinical_diabetes_screening_v1.joblib` (Git-ignored).
- Tracked manifest: `deployment/model/manifest.json`.
- Model version: 1.0.0.
- Artifact SHA-256: `eeff07066a6898af46daceae6b7708dabdc60de15bd0a0c60566ad1a42f9caf8`.
- Analytical-data SHA-256: `8d72ef30fd9d049e03baef7e3283b633b988fef457380880441ff27876cb2435`.
- Aggregate metrics: `reports/model_metrics.json`.

The bundle contains the median imputer, standardizer, estimator, feature order, threshold, outcome wording, version, and safety notice. The API verifies the SHA-256 before loading the trusted joblib output.

## 25. Phase 5 API contract

| Endpoint | Purpose |
|---|---|
| `GET /health` | Process liveness without model access |
| `GET /ready` | Checksummed model readiness |
| `GET /metadata` | Version, model, ordered features, outcome, safety text |
| `POST /v1/predict` | Validated research probability and threshold flag |

Pydantic forbids extra fields and enforces age 20–80, waist 40–200 cm, diastolic BP 20–160 mmHg, and strict boolean activity. Source is `deployment/api/app.py`; the exported contract is `deployment/api/openapi.json`; the non-root container definition is `deployment/api/Dockerfile`. Request bodies are not logged by project code.

## 26. Real packaged-model smoke test

Request:

```json
{"age_years": 50, "waist_cm": 100, "physically_active": true, "diastolic_bp": 75}
```

Observed response through FastAPI/TestClient:

```json
{
  "probability": 0.1035873258,
  "above_validation_threshold": false,
  "threshold": 0.1301215210,
  "model_version": "1.0.0",
  "outcome": "Current diabetes-status research definition",
  "safety_notice": "Educational research only; not a diagnosis, medical advice, or a validated medical device."
}
```

## 27. Implementation corrections and current boundary

- The first API test run found that `deployment` was not on pytest's import path. It is now an explicit package and pytest includes the repository root.
- FastAPI 0.141.1 deprecated its old HTTPX-backed test client; the test dependency was changed to HTTPX2 2.9.1.
- Direct OpenAPI script execution lacked the repository import root. `scripts` is now a package and the stable command is `python -m scripts.export_openapi`.
- The Phase 3 provisional schema is preserved as `configs/app_features_phase3.json`; model validation superseded it with the four-input `configs/app_features.json`.
- External hosting, TLS termination, rate limiting, observability, web UI, and Android UI are not implemented yet.
- Final gate: Ruff passed; 37 files were formatted; strict mypy passed for 5 source files; 10 tests passed; model selection and the locked-test result reproduced; 2 notebooks validated; all 7 raw hashes matched; the artifact hash matched the manifest; the OpenAPI contract exported; and a real API/model smoke prediction returned HTTP 200.

## 28. Next gate

Phase 6 should build the responsive accessible web application from `configs/app_features.json` and `deployment/api/openapi.json`, including validation, review, loading/error states, safe probability communication, and end-to-end tests.

## 29. Phase 6 — Web application

Implemented a responsive, keyboard-accessible, dependency-free client in `web/`. The four controls exactly match model order and API validation: age (years), waist circumference (cm), physical activity (yes/no), and diastolic blood pressure (mmHg). The page includes persistent research-only language, loading/error states, model/threshold context, mobile layout, and reduced-motion support.

FastAPI mounts the static client after its API routes, so one deployment serves both UI and JSON endpoints. Optional cross-origin access is disabled unless exact origins are provided with `CLINICAL_ALLOWED_ORIGINS`. The client has no cookies, analytics, accounts, or application persistence.

In [ ]:
# Local web/API launch (PowerShell)
# .\.venv\Scripts\Activate.ps1
# uvicorn deployment.api.app:app --host 127.0.0.1 --port 8000
# Open http://127.0.0.1:8000 and http://127.0.0.1:8000/docs

### Phase 6 acceptance evidence

- `/` serves the research UI; `/v1/predict` remains the versioned API.
- Browser-side numeric constraints and server-side strict validation agree.
- An API regression test verifies static delivery; Node syntax-checks `web/app.js`.
- The result is framed as a statistical research estimate, never a diagnosis.

## 30. Phase 7 — Native Android application

Implemented a concise Kotlin/Jetpack Compose app in `android/` using the same four inputs and wording. It uses platform `HttpURLConnection` and `org.json`, avoiding a reflection-based networking layer. Only Internet permission is requested; backup is disabled and no health input/result is stored.

Pinned 2026 build foundation: Android Gradle Plugin 9.3.0, Gradle 9.5.0, JDK 17, compile/target SDK 36, Kotlin/Compose compiler 2.3.21, Compose BOM 2026.06.01. SDK 36 is the current generally available Android 16 platform on the clean CI runner. Debug uses the emulator-only host address `http://10.0.2.2:8000` and a debug-only cleartext exception. Release builds require an HTTPS API URL.

In [ ]:
# Android build commands (PowerShell; requires Android Studio/SDK and Gradle)
# gradle -p android :app:assembleDebug
# gradle -p android :app:assembleRelease `
#   -PclinicalApiBaseUrl=https://OWNER-SPACE.hf.space

Local Android compilation was not available on this workstation because Gradle, Android SDK tools, and ADB are not installed. `.github/workflows/android.yml` supplies JDK 17, Android SDK 36/build-tools 36.0.0, and Gradle 9.5.0 to compile the debug APK on every Android change. The first public CI run exposed that platform 37 was not available from `sdkmanager`; pinning the generally available Android 16 SDK 36 corrected it. A retry then stalled in the redundant third-party SDK setup action, so the workflow uses the hosted runner's preinstalled `sdkmanager` directly.

## 31. Phase 8 — Deployment and operations

The production shape is a non-root Python 3.14 Docker container serving FastAPI and the same-origin web client. Access logging is disabled to reduce accidental sensitive-body handling; application code stores no request data. Liveness `/health`, readiness `/ready`, contract `/metadata`, OpenAPI `/docs`, and synthetic prediction checks support operations.

`scripts/build_hf_space.py` assembles a clean ignored directory at `dist/huggingface-space/`. It includes only the Space metadata/Dockerfile, source package, API, web client, project metadata, and checksummed model artifact—never raw or processed participant data. `scripts/publish_hf_space.py` creates or updates a Docker Space through `huggingface_hub`.

In [ ]:
# Local container and Space bundle
# docker build -f deployment/api/Dockerfile -t clinical-ai:1.0.0 .
# docker run --read-only --tmpfs /tmp -p 8000:8000 clinical-ai:1.0.0
# python scripts/build_hf_space.py

# Authenticated Hugging Face publication (never paste token into source/notebook)
# $env:HF_SPACE_ID = "OWNER/clinical-diabetes-screening"
# $env:HF_TOKEN = Read-Host -AsSecureString  # prefer `hf auth login` or CI secret
# python scripts/publish_hf_space.py

CI/CD now has four workflows:

1. `ci.yml`: lint, format, type check, tests, notebook parsing, model SHA verification, web syntax, Space assembly, and Docker build.
2. `android.yml`: clean Android debug compilation.
3. `release.yml`: tagged model checksum verification, GitHub artifact attestation, and release assets.
4. `deploy-huggingface.yml`: protected-environment Docker Space creation/update from `HF_TOKEN` and `HF_SPACE_ID` secrets.

The operations runbook defines smoke tests, body-free monitoring, incident handling, rollback, credential rotation, and versioned model reassessment.

## 32. 7. Version control, reproducibility, and artifact lineage

The detailed policy is saved in `docs/version_control_reproducibility_lineage.md`.

- Protected `main`, short reviewed branches, conventional descriptive commits, semantic release tags, and no secrets/participant data in Git.
- Exact direct dependency/interpreter versions, fixed split seed, executable data/training scripts, clean-runner CI, model card, tests, and notebook validation.
- Lineage: **CDC URL → raw file SHA → cohort/code + processed SHA → fixed split → training configuration → metrics/model card → model SHA → API v1 → web/Android contract → Git commit/tag/CI run → GitHub attestation/Hugging Face commit**.
- The 1.4 KB promoted joblib is release-tracked because it contains fitted preprocessing/model state, not participant rows. `manifest.json` records and the API enforces SHA-256 `eeff07066a6898af46daceae6b7708dabdc60de15bd0a0c60566ad1a42f9caf8`.
- A changed cohort, feature contract, dependency, threshold, fitted state, or API schema requires regeneration, revalidation, a version update, and a changelog entry.

In [ ]:
# First authenticated GitHub publication
# git remote add origin https://github.com/OWNER/clinical-ai-portfolio.git
# git push -u origin main
# git tag -s v1.0.0 -m "Clinical AI portfolio v1.0.0"
# git push origin v1.0.0

# Verify the tagged GitHub provenance after release
# gh attestation verify deployment/model/clinical_diabetes_screening_v1.joblib `
#   -R OWNER/clinical-ai-portfolio

## 33. 8. Copyright, licensing, attribution, and privacy

The complete decision record is `docs/copyright_licensing_attribution_privacy.md`; user-facing terms are in `LICENSE`, `NOTICE`, `THIRD_PARTY_NOTICES.md`, `CITATION.cff`, and `PRIVACY.md`.

- Original repository work: copyright 2026 contributors, MIT licensed, no warranty or clinical-fitness claim.
- NHANES data/docs/marks are not relicensed. Preserve exact CDC/NCHS attribution, current public-use terms, and a clear non-endorsement statement.
- Keep the model artifact paired with its manifest, model card, source attribution, limitations, intended-use restrictions, and safety notice.
- Review transitive dependency licenses before release; package availability is not a license conclusion.
- Never commit or deploy participant rows. Do not attempt re-identification. Accept only four structured values, reject extra fields, use HTTPS, avoid request-body logs/analytics, minimize platform retention, and publish the actual operator contact/policy.
- This remains educational research—not diagnosis, prognosis, treatment advice, an emergency service, regulatory approval, or a medical device claim.

## 34. How to use the API and applications

Full instructions are in `docs/user_guide.md`.

**API:** start Uvicorn, inspect `/docs`, POST the four fields to `/v1/predict`, and treat the response threshold flag only as model-evaluation context. HTTP 422 indicates contract/range errors; HTTP 503 indicates a model-readiness problem.

**Web:** open `/`, enter values in the units/ranges shown, submit once, and read the probability together with the model version and warning. No application data are saved.

**Android:** run the debug app in an emulator while the local API is active; build release only with the final HTTPS Space URL. A physical phone needs an authorized reachable HTTPS endpoint.

In [ ]:
# Example API request (PowerShell)
# $body = @{age_years=50; waist_cm=100; physically_active=$true; diastolic_bp=75} | ConvertTo-Json
# Invoke-RestMethod -Method Post -Uri http://127.0.0.1:8000/v1/predict `
#   -ContentType application/json -Body $body

## 35. Publication status and final Phase 6–8 gate

Publication completed without writing credentials into source. The audited baseline commit `20aa229` was pushed to the public repository [bearstree/clinical-ai-portfolio](https://github.com/bearstree/clinical-ai-portfolio). The clean Docker Space bundle was uploaded to [weiyi/clinical-diabetes-screening](https://huggingface.co/spaces/weiyi/clinical-diabetes-screening); the source-linked deployment commit is `92787c8ecdb0e63480340686ac88e8ae0e85455c`.

The live HTTPS service returned `status=ok` from `/health`, `status=ready` from `/ready`, and model `1.0.0` probability `0.103587325784365` (below threshold `0.13012152100670846`) for the documented synthetic input. The workstation Docker daemon and Android/Gradle toolchain were unavailable, so clean Docker and Android builds are enforced by GitHub Actions rather than claimed as local results.

Verified locally on 2026-08-19:

- Ruff: all checks passed; 49 Python files formatted.
- mypy: no issues in 5 source files.
- pytest: 11 passed, including the web delivery test.
- Node: `web/app.js` syntax valid.
- Space: clean bundle assembled, uploaded, and live-smoke-tested successfully.
- Model: tracked artifact SHA-256 matches the manifest.
- Notebook: parsed and validated after this append.